# Heatmap Temporal — Inversion Publica en el Peru (2017-2026)

**Fuente:** MEF - Banco de Inversiones INVIERTE.PE  
**Intensidad:** Monto viable (S/.)  
**Autor:** Leo Molina (@leydimolina20)

---
Ejecuta las celdas en orden. El mapa final se descarga como HTML y se abre en tu navegador.

In [ ]:
!pip install folium --quiet

In [ ]:
import requests, io, gzip, warnings
import pandas as pd
warnings.filterwarnings('ignore')

print('Descargando dataset...')
BASE = 'https://raw.githubusercontent.com/leydimolina20/peru-investment-headmap/main/data'
raw = requests.get(f'{BASE}/inversiones_historico.gz').content
df = pd.read_csv(io.BytesIO(gzip.decompress(raw)), low_memory=False)

print(f'Proyectos: {len(df):,}')
print(f'Anos: {int(df["ANIO"].min())} - {int(df["ANIO"].max())}')
print(f'Monto total: S/. {df["MONTO_VIABLE"].sum()/1e9:.1f} mil millones')
resumen = df.groupby('ANIO').agg(
    proyectos=('MONTO_VIABLE','count'),
    monto_MM=('MONTO_VIABLE', lambda x: round(x.sum()/1e6, 1))
).reset_index()
print(resumen.to_string(index=False))

In [ ]:
import folium
from folium.plugins import HeatMapWithTime, Fullscreen, MiniMap

YEARS = sorted(df['ANIO'].dropna().unique().astype(int))

monto_cap = df['MONTO_VIABLE'].quantile(0.95)
df['peso'] = (df['MONTO_VIABLE'].clip(upper=monto_cap) / monto_cap).clip(lower=0.2)

heat_data = []
for yr in YEARS:
    sub = df[df['ANIO'] == yr]
    heat_data.append(sub[['LATITUD','LONGITUD','peso']].values.tolist())

m = folium.Map(location=[-9.19, -75.0], zoom_start=6, tiles='CartoDB dark_matter')

HeatMapWithTime(
    data=heat_data,
    index=[str(y) for y in YEARS],
    radius=20,
    blur=15,
    max_opacity=0.9,
    min_opacity=0.4,
    use_local_extrema=True,
    gradient={0.2:'#1565C0', 0.4:'#26A69A', 0.6:'#FFC107', 0.8:'#FF5722', 1.0:'#B71C1C'},
    auto_play=True,
    display_index=True,
    speed_step=0.1,
).add_to(m)

Fullscreen(position='topright', title='Pantalla completa', title_cancel='Salir', force_separate_button=True).add_to(m)
MiniMap(toggle_display=True, position='bottomleft').add_to(m)

m.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;top:10px;left:50px;z-index:9999;background:rgba(0,0,0,0.80);padding:12px 18px;border-radius:8px;color:white;font-family:Arial;">'
    '<h3 style="margin:0;font-size:17px">Expansion de la Inversion Publica en el Peru</h3>'
    '<p style="margin:4px 0 0 0;font-size:11px;color:#90CAF9">2017-2026 · 130,884 proyectos · Intensidad = Monto viable (S/.)</p>'
    '</div>'
))
m.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;top:10px;right:50px;z-index:9999;color:#ccc;font-family:Arial;font-size:11px;background:rgba(0,0,0,0.55);padding:6px 10px;border-radius:5px">'
    'Fuente: MEF - INVIERTE.PE | Elaborado por: Leo Molina'
    '</div>'
))
m.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:120px;right:10px;z-index:9999;background:rgba(0,0,0,0.75);padding:10px 14px;border-radius:7px;color:white;font-family:Arial;font-size:11px">'
    '<b>Intensidad de inversion</b><br>'
    '<span style="color:#B71C1C">&#9632;</span> Muy alta<br>'
    '<span style="color:#FF5722">&#9632;</span> Alta<br>'
    '<span style="color:#FFC107">&#9632;</span> Media<br>'
    '<span style="color:#26A69A">&#9632;</span> Baja<br>'
    '<span style="color:#1565C0">&#9632;</span> Muy baja'
    '</div>'
))

# Guardar HTML
m.save('inversiones_heatmap_temporal.html')
print('Mapa guardado!')

In [ ]:
# Descargar el HTML desde Colab a tu PC
from google.colab import files
files.download('inversiones_heatmap_temporal.html')
print('Abre el archivo descargado en tu navegador (Chrome o Firefox)')